# ToolStrategy와 ProviderStrategy

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3 pillow

Note: you may need to restart the kernel to use updated packages.


## 환경 준비

In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("langchain", version("langchain"))
print("langgraph", version("langgraph"))
print("API 키 존재 여부 확인 완료")

langchain 1.4.0
langgraph 1.2.11
API 키 존재 여부 확인 완료


## 출력 스키마와 두 에이전트

In [3]:
# 같은 PolicyAnswer 스키마를 ToolStrategy와 ProviderStrategy에 각각 연결합니다.
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy
class PolicyAnswer(BaseModel):
    topic: str = Field(description="규정 주제")
    answer: str = Field(description="근거에 맞는 답")
    source: str = Field(description="자료 식별자")
tool_agent = create_agent("openai:gpt-5.4-mini", [], response_format=ToolStrategy(PolicyAnswer))
provider_agent = create_agent("openai:gpt-5.4-mini", [], response_format=ProviderStrategy(PolicyAnswer))
prompt = "자료 R-14에 따르면 일반 도서 대출 기간은 14일입니다. 구조화해 주세요."

## 두 전략 실제 실행

In [4]:
# 같은 입력으로 두 전략을 호출하고 structured_response의 세 필드를 검증합니다.
rows = {}
for name, current in [("tool", tool_agent), ("provider", provider_agent)]:
    result = current.invoke({"messages": [{"role": "user", "content": prompt}]})
    parsed = result["structured_response"]
    rows[name] = {"parsed": parsed.model_dump(), "messages": len(result["messages"]), "ai_messages": sum(type(m).__name__ == "AIMessage" for m in result["messages"])}
    print(name, rows[name])
    assert "R-14" in parsed.source and "14" in parsed.answer

tool {'parsed': {'topic': '일반 도서 대출 기간', 'answer': '일반 도서의 대출 기간은 14일입니다.', 'source': 'R-14'}, 'messages': 3, 'ai_messages': 1}


provider {'parsed': {'topic': '일반 도서 대출 기간', 'answer': '일반 도서의 대출 기간은 14일입니다.', 'source': '자료 R-14'}, 'messages': 2, 'ai_messages': 1}


## 메인 루프 메시지 확인

In [5]:
# AIMessage 수와 전체 메시지 수를 나란히 출력해 추가 모델 호출과 추가 메시지를 구분합니다.
print(rows)
print("구조화 결과 키:", "structured_response")
assert rows["tool"]["ai_messages"] >= 1 and rows["provider"]["ai_messages"] >= 1

{'tool': {'parsed': {'topic': '일반 도서 대출 기간', 'answer': '일반 도서의 대출 기간은 14일입니다.', 'source': 'R-14'}, 'messages': 3, 'ai_messages': 1}, 'provider': {'parsed': {'topic': '일반 도서 대출 기간', 'answer': '일반 도서의 대출 기간은 14일입니다.', 'source': '자료 R-14'}, 'messages': 2, 'ai_messages': 1}}
구조화 결과 키: structured_response
